# Planetary Computer (MPC)

Description:
- Planetary scale Earth Observation platform

Resources:
- https://planetarycomputer.microsoft.com/
- https://planetarycomputer.microsoft.com/catalog
- https://github.com/Microsoft/planetary-computer-apis
- https://github.com/microsoft/planetary-computer-apis/blob/main/docs/collection-config.md
- https://planetarycomputer.microsoft.com/explore
- https://planetarycomputer.microsoft.com/explore?c=30.0586%2C29.9930&z=2.00&v=2&d=landsat-c2-l2&s=false%3A%3A100%3A%3Atrue&ae=0&sr=desc&m=Most+recent+%28low+cloud%29&r=Natural+color

## Example 1

Work with datasets in MPC and create stac complient assets

In [ ]:
import pystac
import planetary_computer
import rioxarray as rxr

item_url = "https://planetarycomputer.microsoft.com/api/stac/v1/collections/landsat-c2-l2/items/LC09_L2SP_217066_20231126_02_T1"

# Load the individual item metadata and sign the assets
item = pystac.Item.from_file(item_url)
signed_item = planetary_computer.sign(item)

# Open one of the data assets (other asset keys to use: 'red', 'blue', 'drad', 'emis', 'emsd', 'trad', 'urad', 'atran', 'cdist', 'green', 'nir08', 'lwir11', 'swir16', 'swir22', 'coastal', 'qa_pixel', 'qa_radsat', 'qa_aerosol')
asset_href = signed_item.assets["qa"].href
ds = rxr.open_rasterio(asset_href)
ds

## Example 2

Search the MPC catalog and open the dataset

In [ ]:
from pystac_client import Client
import planetary_computer as pc
import rioxarray as rxr

# Search against the Planetary Computer STAC API
catalog = Client.open(
  "https://planetarycomputer.microsoft.com/api/stac/v1"
)

# Define your area of interest
aoi = {
  "type": "Polygon",
  "coordinates": [
    [
      [-56.60156250000078, -39.97712009843911],
      [116.71875000000045, -39.97712009843911],
      [116.71875000000045, 72.31578530526212],
      [-56.60156250000078, 72.31578530526212],
      [-56.60156250000078, -39.97712009843911]
    ]
  ]
}

# Define your search with CQL2 syntax
search = catalog.search(filter_lang="cql2-json", filter={
  "op": "and",
  "args": [
    {"op": "s_intersects", "args": [{"property": "geometry"}, aoi]},
    {"op": "=", "args": [{"property": "collection"}, "landsat-c2-l2"]},
    {"op": "<=", "args": [{"property": "eo:cloud_cover"}, 10]}
  ]
})

# Grab the first item from the search results and sign the assets
first_item = next(search.items())
signed_item = pc.sign_item(first_item).assets

asset_href = signed_item["coastal"].href
ds = rxr.open_rasterio(asset_href)
ds